In [1]:
import pandas as pd

def read_csv_to_dataframe(file_path: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(file_path)
        return df
    except Exception as e:
        print(f"Error reading CSV file: {e}")
        return pd.DataFrame()

In [2]:
Rosedata_df = read_csv_to_dataframe("Event_core_v4.csv")

In [3]:
print(Rosedata_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46 entries, 0 to 45
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   event_date                    29 non-null     object 
 1   event_id                      29 non-null     object 
 2   parent_event_id               46 non-null     object 
 3   type                          46 non-null     object 
 4   location_id                   46 non-null     object 
 5   decimal_latitud               29 non-null     float64
 6   decimal_longitude             29 non-null     float64
 7   depth_m                       46 non-null     int64  
 8   habitat                       46 non-null     object 
 9   conteo_ind                    46 non-null     int64  
 10  temperat_grad_c               46 non-null     float64
 11  total_organic_matter_percent  46 non-null     float64
 12  proport_silt_clay_percent     46 non-null     int64  
 13  prop_si

NOTA: ESTE EJERCICIO ES SOLO UN EJEMPLO PORQUE ESTOS DATOS NO TIENE DISTRIBUCIÓN NORMAL

In [4]:
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

def two_way_anova_tukey(df, response, factor1, factor2, alpha=0.05):
    """
    Performs a two-way ANOVA and Tukey HSD post-hoc test if significant effects are found.

    Parameters:
    - df (DataFrame): The dataset.
    - response (str): Dependent variable column.
    - factor1 (str): First independent variable column.
    - factor2 (str): Second independent variable column.
    - alpha (float): Significance level (default = 0.05).
    """
    # Fit the two-way ANOVA model and generate the ANOVA table
    model = ols(f'{response} ~ C({factor1}) + C({factor2}) + C({factor1}):C({factor2})', data=df).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)
    print("\nANOVA Results:\n", anova_table)

    # Run Tukey HSD post-hoc test if significant effects are found
    if (anova_table['PR(>F)'] < alpha).any():
        print("\nSignificant effects detected. Running Tukey HSD post-hoc test:")
        df['Group'] = df[factor1].astype(str) + " - " + df[factor2].astype(str)
        tukey = pairwise_tukeyhsd(df[response], df['Group'], alpha=alpha)
        print(tukey.summary())
    else:
        print("\nNo significant effects detected. Post-hoc test is not required.")

In [5]:
two_way_anova_tukey(Rosedata_df, response="temperat_grad_c", factor1="parent_event_id", factor2="habitat")


ANOVA Results:
                                  sum_sq    df          F        PR(>F)
C(parent_event_id)             9.080424   2.0  23.710891  1.617360e-07
C(habitat)                     0.886456   1.0   4.629445  3.751972e-02
C(parent_event_id):C(habitat)  5.234405   2.0  13.668129  2.993782e-05
Residual                       7.659286  40.0        NaN           NaN

Significant effects detected. Running Tukey HSD post-hoc test:
              Multiple Comparison of Means - Tukey HSD, FWER=0.05               
       group1               group2        meandiff p-adj   lower   upper  reject
--------------------------------------------------------------------------------
     GB - coral reef GB - seagrass meadow    -0.85 0.0197  -1.606  -0.094   True
     GB - coral reef      GG - coral reef   0.2071 0.9557 -0.5213  0.9356  False
     GB - coral reef GG - seagrass meadow      0.9 0.0036  0.2239  1.5761   True
     GB - coral reef      GM - coral reef   0.2071 0.9557 -0.5213  0.9356  Fal